In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_DIR = "/Volumes/workspace/default/my_volume/drone_pipeline/data/bronze"
SILVER_DIR = "/Volumes/workspace/default/my_volume/drone_pipeline/data/silver"

In [0]:
def get_spark():
    return SparkSession.builder.appName("DroneSilverLayer").getOrCreate()

In [0]:
def clean_drones(spark):
    df = spark.read.parquet(f"{BRONZE_DIR}/drones")
    silver = (
        df.dropna(subset=["drone_id", "model"])
        .dropDuplicates(["drone_id"])
        .withColumn("max_range_km", F.col("max_range_km").cast("float"))
        .withColumn("processed_time", F.current_timestamp())
        .select("drone_id", "model", "max_range_km", "processed_time")
    )
    silver.write.format("parquet").mode("overwrite").save(f"{SILVER_DIR}/drones")
    print(f"[SILVER] drones -> {silver.count()} rows")
    return silver

In [0]:
def clean_deliveries(spark):
    df = spark.read.parquet(f"{BRONZE_DIR}/deliveries")
    silver = (
        df.dropna(subset=["delivery_id", "drone_id", "distance_km"])
        .dropDuplicates(["delivery_id"])
        .withColumn("start_time", F.to_timestamp("start_time"))
        .withColumn("end_time", F.to_timestamp("end_time"))
        .withColumn("distance_km", F.col("distance_km").cast("float"))
        .withColumn("battery_consumed", F.col("battery_consumed").cast("float"))

        # derived column: delivery_duration in minutes
        
        .withColumn(
            "delivery_duration",
            (F.col("end_time").cast("long") - F.col("start_time").cast("long")) / 60.0,
        )
        .withColumn("processed_time", F.current_timestamp())
        .select(
            "delivery_id", "drone_id", "source", "destination", "distance_km",
            "start_time", "end_time", "delivery_duration", "battery_consumed",
            "processed_time",
        )
    )
    silver.write.format("parquet").mode("overwrite").save(f"{SILVER_DIR}/deliveries")
    print(f"[SILVER] deliveries -> {silver.count()} rows")
    return silver

In [0]:
def clean_flight_logs(spark, deliveries_silver):
    df = spark.read.parquet(f"{BRONZE_DIR}/flight_logs")
    logs = (
        df.dropna(subset=["log_id", "delivery_id", "status"])
        .dropDuplicates(["log_id"])
        .withColumn("battery_level", F.col("battery_level").cast("float"))
        .withColumn("gps_signal", F.col("gps_signal").cast("float"))

        # derived column: failure_flag

        .withColumn(
            "failure_flag",
            F.when(F.col("status").startswith("FAILED"), F.lit(1)).otherwise(F.lit(0)),
        )
        .withColumn("processed_time", F.current_timestamp())
    )
    enriched = logs.join(
        deliveries_silver.select("delivery_id", "destination"),
        on="delivery_id",
        how="left",
    ).select(
        "log_id", "delivery_id", "drone_id", "destination",
        "battery_level", "gps_signal", "status", "failure_flag", "processed_time",
    )

    enriched.write.format("parquet").mode("overwrite").save(f"{SILVER_DIR}/flight_logs")
    print(f"[SILVER] flight_logs -> {enriched.count()} rows")
    return enriched


In [0]:
def main():
    spark = get_spark()
    try:
        spark.sparkContext.setLogLevel("ERROR")
    except Exception:
        pass

    clean_drones(spark)
    deliveries_silver = clean_deliveries(spark)
    clean_flight_logs(spark, deliveries_silver)

    print("\nSilver layer complete. Data is cleaned, typed, joined, and enriched.")
    spark.stop()


if __name__ == "__main__":
    main()


[SILVER] drones -> 25 rows
[SILVER] deliveries -> 600 rows
[SILVER] flight_logs -> 600 rows

Silver layer complete. Data is cleaned, typed, joined, and enriched.
